# COMP8240 · LLM judge on the app reviews (about 10 minutes)

The judge from notebook C (Qwen2.5-7B-Instruct, 4-bit, run on this GPU; prompt v1), applied to the review
test split rebuilt from the 9 October outputs: 346 items (meaning, disclosure and residual, for the
original reviews and their SanText version at ε = 3). The items and the judge script are embedded below,
so nothing needs uploading and Google Drive isn't used. The items are built by `scripts/prepare_reviews_judge.py`; in this public copy one person's name is replaced by [NAME] (the run used the original text).

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.**
3. At the end, `reviews_judge_outputs.zip` downloads. Upload it in the chat.

In [ ]:
# Step 1 · GPU check, the judge's one extra package, and the embedded files
import base64, gzip, json, os, platform, subprocess, sys
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "bitsandbytes"], check=False)
os.makedirs("judge_reviews", exist_ok=True)
ITEMS_B64 = "..."
SRC_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKbGxtX2p1ZGdlLnB5OiBMTE0tYXMtanVkZ2UgZm9yIHNhbml0aXplZCB0ZXh0LCBydW4gbG9jYWxseSB3aXRoIGFuIG9wZW4td2VpZ2h0cyBtb2RlbC4KClRoZSBtb2RlbCBuZXZlciB3cml0ZXMgZnJlZSB0ZXh0LiBGb3IgZWFjaCBwcm9tcHQgd2UgcmVhZCBpdHMgcHJvYmFiaWxpdGllcyBmb3IgdGhlIGFsbG93ZWQgYW5zd2VyCnRva2VucyBhdCB0aGUgZmlyc3QgcG9zaXRpb24gb2YgaXRzIHJlcGx5LCBzbyBldmVyeSBqdWRnbWVudCBpcyBkZXRlcm1pbmlzdGljLCBjaGVhcCBhbmQgY29tZXMgd2l0aAphIGNvbmZpZGVuY2U6CiAgbWVhbmluZyAgICAgb3JpZ2luYWwgKyBzYW5pdGl6ZWQgdGV4dCAtPiBob3cgd2VsbCBpcyB0aGUgbWVhbmluZyBwcmVzZXJ2ZWQ/IDEtNQogICAgICAgICAgICAgIHNjb3JlID0gZXhwZWN0ZWQgdmFsdWUgb3ZlciB0aGUgcHJvYmFiaWxpdGllcyBvZiB0aGUgdG9rZW5zICIxIi4uIjUiCiAgZGlzY2xvc3VyZSAgb25lIHJldmlldyAtPiBkb2VzIGl0IHJldmVhbCBwZXJzb25hbCBpbmZvcm1hdGlvbiBhYm91dCBpdHMgd3JpdGVyPyB5ZXMvbm8KICAgICAgICAgICAgICAodGhlIHNhbWUgZGVmaW5pdGlvbiBhcyB0aGUgaHVtYW4gYW5ub3RhdGlvbiBndWlkZWxpbmVzKQogIHJlc2lkdWFsICAgIHNhbml0aXplZCByZXZpZXcgKyB0aGUgaW5mb3JtYXRpb24gaXRzIG9yaWdpbmFsIGRpc2Nsb3NlZCAtPiBpcyBpdCBzdGlsbCByZXZlYWxlZD8geWVzL25vCiAgICAgICAgICAgICAgcF95ZXMgPSBQKHllcykgLyAoUCh5ZXMpICsgUChubykpLCBzdW1taW5nICJ5ZXMiLyJZZXMiIGFuZCAibm8iLyJObyIKZm9ybWF0X21hc3MgcmVjb3JkcyBob3cgbXVjaCBvZiB0aGUgbW9kZWwncyBwcm9iYWJpbGl0eSBmYWxscyBvbiB0aGUgYWxsb3dlZCBhbnN3ZXIgdG9rZW5zIGF0IGFsbAooY2xvc2UgdG8gMSBtZWFucyB0aGUgbW9kZWwgaXMgZm9sbG93aW5nIHRoZSBhbnN3ZXIgZm9ybWF0KS4KCk5vdGhpbmcgbGVhdmVzIHRoZSBtYWNoaW5lOiB0aGUgbW9kZWwgaXMgZG93bmxvYWRlZCBvbmNlIGFuZCBydW4gb24gdGhlIGxvY2FsIEdQVS4KClVzYWdlCiAgcHl0aG9uIGxsbV9qdWRnZS5weSAtLWl0ZW1zIGp1ZGdlX2l0ZW1zLmpzb25sIC0tb3V0IGp1ZGdlX3Jlc3VsdHMuanNvbmwKICAgICAgWy0tbW9kZWwgUXdlbi9Rd2VuMi41LTdCLUluc3RydWN0IC0tZmFsbGJhY2sgUXdlbi9Rd2VuMi41LTNCLUluc3RydWN0IC0tYmF0Y2hfc2l6ZSA4XQpSZXN1bWFibGU6IGl0ZW1zIGFscmVhZHkgaW4gLS1vdXQgYXJlIHNraXBwZWQuCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aW1lCgpQUk9NUFRfVkVSU0lPTiA9ICJ2MSIKU1lTVEVNID0gKCJZb3UgYXJlIGEgY2FyZWZ1bCBhbm5vdGF0b3IgaW4gYSByZXNlYXJjaCBzdHVkeSBvbiB0ZXh0IHByaXZhY3kuICIKICAgICAgICAgICJGb2xsb3cgdGhlIGluc3RydWN0aW9ucyBleGFjdGx5IGFuZCBhbnN3ZXIgd2l0aCBhIHNpbmdsZSB0b2tlbi4iKQoKTUVBTklORyA9ICIiIlRleHQgQSBpcyBhbiBvcmlnaW5hbCB0ZXh0LiBUZXh0IEIgd2FzIG1hZGUgZnJvbSBUZXh0IEEgYnkgcmVwbGFjaW5nIHNvbWUgb2YgaXRzIHdvcmRzIHdpdGggb3RoZXIgd29yZHMuCkhvdyB3ZWxsIGRvZXMgVGV4dCBCIHByZXNlcnZlIHRoZSBtZWFuaW5nIG9mIFRleHQgQT8KMSA9IHVucmVsYXRlZCBvciB1bmludGVsbGlnaWJsZQoyID0gbW9zdGx5IGxvc3Q7IG9ubHkgZnJhZ21lbnRzIHN1cnZpdmUKMyA9IHRoZSB0b3BpYyBzdXJ2aXZlcywgYnV0IGltcG9ydGFudCBkZXRhaWxzIGFyZSBsb3N0IG9yIGNoYW5nZWQKNCA9IG1vc3RseSB0aGUgc2FtZSBtZWFuaW5nOyBtaW5vciBkZXRhaWxzIGFyZSBsb3N0CjUgPSB0aGUgc2FtZSBtZWFuaW5nCgpUZXh0IEE6IDw8PHthfT4+PgpUZXh0IEI6IDw8PHtifT4+PgoKQW5zd2VyIHdpdGggb25lIGRpZ2l0IGZyb20gMSB0byA1LiIiIgoKRElTQ0xPU1VSRSA9ICIiIkRvZXMgdGhlIGZvbGxvd2luZyBhcHAgcmV2aWV3IHJldmVhbCBwZXJzb25hbCBpbmZvcm1hdGlvbiBhYm91dCBpdHMgd3JpdGVyLCBvciBhYm91dCBhbm90aGVyIGlkZW50aWZpYWJsZSBwZXJzb24sIHRoYXQgdGhleSBtaWdodCBub3Qgd2FudCBsaW5rZWQgdG8gdGhlbT8KUGVyc29uYWwgaW5mb3JtYXRpb24gbWVhbnM6Ci0gSEVBTFRIOiBhIGNvbmRpdGlvbiwgc3ltcHRvbSwgbWVkaWNhdGlvbiwgdHJlYXRtZW50LCBwcmVnbmFuY3kgb3IgbWVudGFsIGhlYWx0aAotIEZJTkFOQ0U6IGFtb3VudHMsIGRlYnRzLCBsb2FucywgaW5jb21lLCBhY2NvdW50IHByb2JsZW1zLCBvciB3aGljaCBiYW5rIHRoZXkgdXNlCi0gTE9DQVRJT046IHN1YnVyYiwgYWRkcmVzcywgd29ya3BsYWNlIG9yIGJyYW5jaAotIElERU5USVRZOiBhIG5hbWUsIGFnZSwgZmFtaWx5IG1lbWJlciBvciBqb2IKLSBDT05UQUNUOiBhbiBlbWFpbCwgcGhvbmUgbnVtYmVyLCBhY2NvdW50IG9yIHJlZmVyZW5jZSBudW1iZXIKT3BpbmlvbnMgYWJvdXQgdGhlIGFwcCwgaXRzIGZlYXR1cmVzIG9yIGl0cyBwcmljZSwgYW5kIGdlbmVyaWMgdXNlICgiSSB1c2UgaXQgZGFpbHkiLCAibXkgbWVkcyIsICJteSBhY2NvdW50IikgZG8gbm90IGNvdW50LgoKUmV2aWV3OiA8PDx7YX0+Pj4KCkFuc3dlciB5ZXMgb3Igbm8uIiIiCgpSRVNJRFVBTCA9ICIiIkFuIG9yaWdpbmFsIGFwcCByZXZpZXcgZGlzY2xvc2VkIHRoaXMgcGVyc29uYWwgaW5mb3JtYXRpb24gYWJvdXQgaXRzIHdyaXRlcjoge2luZm99LgpUaGUgdGV4dCBiZWxvdyB3YXMgbWFkZSBmcm9tIHRoYXQgcmV2aWV3IGJ5IHJlcGxhY2luZyBzb21lIG9mIGl0cyB3b3JkcyB3aXRoIG90aGVyIHdvcmRzLgpEb2VzIHRoZSB0ZXh0IGJlbG93IHN0aWxsIHJldmVhbCB0aGF0IGluZm9ybWF0aW9uLCBvciBlbm91Z2ggb2YgaXQgdGhhdCBhIHJlYWRlciBjb3VsZCBpbmZlciBpdD8KClRleHQ6IDw8PHtifT4+PgoKQW5zd2VyIHllcyBvciBuby4iIiIKCgpkZWYgY2xpcF93b3Jkcyh0ZXh0LCBuKToKICAgIHdvcmRzID0gKHRleHQgb3IgIiIpLnNwbGl0KCkKICAgIHJldHVybiAiICIuam9pbih3b3Jkc1s6bl0pICsgKCIgWy4uLl0iIGlmIGxlbih3b3JkcykgPiBuIGVsc2UgIiIpCgoKZGVmIGJ1aWxkX3Byb21wdChpdGVtLCBtYXhfd29yZHMpOgogICAgYSwgYiA9IGNsaXBfd29yZHMoaXRlbS5nZXQoImEiKSwgbWF4X3dvcmRzKSwgY2xpcF93b3JkcyhpdGVtLmdldCgiYiIpLCBtYXhfd29yZHMpCiAgICBpZiBpdGVtWyJ0YXNrIl0gPT0gIm1lYW5pbmciOgogICAgICAgIHJldHVybiBNRUFOSU5HLmZvcm1hdChhPWEsIGI9YikKICAgIGlmIGl0ZW1bInRhc2siXSA9PSAiZGlzY2xvc3VyZSI6CiAgICAgICAgcmV0dXJuIERJU0NMT1NVUkUuZm9ybWF0KGE9YSkKICAgIGlmIGl0ZW1bInRhc2siXSA9PSAicmVzaWR1YWwiOgogICAgICAgIHJldHVybiBSRVNJRFVBTC5mb3JtYXQoaW5mbz1pdGVtWyJpbmZvIl0sIGI9YikKICAgIHJhaXNlIFZhbHVlRXJyb3IoaXRlbVsidGFzayJdKQoKCmRlZiBhbnN3ZXJfdG9rZW5faWRzKHRva2VuaXplcik6CiAgICBkZWYgZmlyc3RfaWQocyk6CiAgICAgICAgaWRzID0gdG9rZW5pemVyLmVuY29kZShzLCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpCiAgICAgICAgcmV0dXJuIGlkc1swXSBpZiBpZHMgZWxzZSBOb25lCiAgICBkaWdpdHMgPSB7ZDogZmlyc3RfaWQoZCkgZm9yIGQgaW4gIjEyMzQ1In0KICAgIHllcyA9IHNvcnRlZCh7aSBmb3IgaSBpbiAoZmlyc3RfaWQoInllcyIpLCBmaXJzdF9pZCgiWWVzIikpIGlmIGkgaXMgbm90IE5vbmV9KQogICAgbm8gPSBzb3J0ZWQoe2kgZm9yIGkgaW4gKGZpcnN0X2lkKCJubyIpLCBmaXJzdF9pZCgiTm8iKSkgaWYgaSBpcyBub3QgTm9uZX0pCiAgICByZXR1cm4gZGlnaXRzLCB5ZXMsIG5vCgoKZGVmIGxvYWRfbW9kZWwobW9kZWxfaWQsIHF1YW50aXplLCB0b3JjaCk6CiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsRm9yQ2F1c2FsTE0sIEF1dG9Ub2tlbml6ZXIKICAgIHRvayA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX2lkKQogICAgdG9rLnBhZGRpbmdfc2lkZSA9ICJsZWZ0IgogICAgaWYgdG9rLnBhZF90b2tlbiBpcyBOb25lOgogICAgICAgIHRvay5wYWRfdG9rZW4gPSB0b2suZW9zX3Rva2VuCiAgICBrd2FyZ3MgPSB7ImRldmljZV9tYXAiOiAiYXV0byJ9CiAgICBpZiBxdWFudGl6ZToKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQml0c0FuZEJ5dGVzQ29uZmlnCiAgICAgICAga3dhcmdzWyJxdWFudGl6YXRpb25fY29uZmlnIl0gPSBCaXRzQW5kQnl0ZXNDb25maWcobG9hZF9pbl80Yml0PVRydWUsIGJuYl80Yml0X3F1YW50X3R5cGU9Im5mNCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYm5iXzRiaXRfY29tcHV0ZV9kdHlwZT10b3JjaC5mbG9hdDE2KQogICAgdHJ5OgogICAgICAgIG1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKG1vZGVsX2lkLCBkdHlwZT10b3JjaC5mbG9hdDE2LCAqKmt3YXJncykKICAgIGV4Y2VwdCBUeXBlRXJyb3I6ICAjIG9sZGVyIHRyYW5zZm9ybWVycyB1c2UgdG9yY2hfZHR5cGUKICAgICAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZChtb2RlbF9pZCwgdG9yY2hfZHR5cGU9dG9yY2guZmxvYXQxNiwgKiprd2FyZ3MpCiAgICByZXR1cm4gdG9rLCBtb2RlbC5ldmFsKCkKCgpkZWYgZGVjb2Rlcl9hbmRfaGVhZChtb2RlbCk6CiAgICAiIiJUaGUgdHJhbnNmb3JtZXIgYm9keSBhbmQgdGhlIG91dHB1dCBsYXllciwgc28gb25seSB0aGUgbGFzdCBwb3NpdGlvbidzIGxvZ2l0cyBhcmUgY29tcHV0ZWQuIiIiCiAgICB0cnk6CiAgICAgICAgZGVjb2RlciA9IG1vZGVsLmdldF9kZWNvZGVyKCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGVjb2RlciA9IE5vbmUKICAgIGlmIGRlY29kZXIgaXMgTm9uZToKICAgICAgICBkZWNvZGVyID0gZ2V0YXR0cihtb2RlbCwgIm1vZGVsIiwgTm9uZSkgb3IgbW9kZWwuYmFzZV9tb2RlbAogICAgaGVhZCA9IG1vZGVsLmdldF9vdXRwdXRfZW1iZWRkaW5ncygpCiAgICBpZiBoZWFkIGlzIE5vbmU6CiAgICAgICAgaGVhZCA9IG1vZGVsLmxtX2hlYWQKICAgIHJldHVybiBkZWNvZGVyLCBoZWFkCgoKZGVmIGxhc3RfcG9zaXRpb25fbG9naXRzKGRlY29kZXIsIGhlYWQsIGVuYyk6CiAgICAiIiJMb2dpdHMgb2YgdGhlIG5leHQgdG9rZW4gb25seSAodGhlIGZ1bGwgbG9naXRzIHRlbnNvciB3b3VsZCBub3QgZml0IGZvciBhIGJhdGNoKS4iIiIKICAgIGF0dGVudGlvbiA9IGVuY1siYXR0ZW50aW9uX21hc2siXQogICAgcG9zaXRpb25faWRzID0gKGF0dGVudGlvbi5jdW1zdW0oLTEpIC0gMSkuY2xhbXAobWluPTApCiAgICBoaWRkZW4gPSBkZWNvZGVyKGlucHV0X2lkcz1lbmNbImlucHV0X2lkcyJdLCBhdHRlbnRpb25fbWFzaz1hdHRlbnRpb24sCiAgICAgICAgICAgICAgICAgICAgIHBvc2l0aW9uX2lkcz1wb3NpdGlvbl9pZHMpLmxhc3RfaGlkZGVuX3N0YXRlWzosIC0xLCA6XQogICAgcmV0dXJuIGhlYWQoaGlkZGVuKS5mbG9hdCgpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkxvY2FsIExMTS1hcy1qdWRnZSBmb3Igc2FuaXRpemVkIHRleHQuIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1pdGVtcyIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tb2RlbCIsIGRlZmF1bHQ9IlF3ZW4vUXdlbjIuNS03Qi1JbnN0cnVjdCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcXVhbnRpemUiLCBkZWZhdWx0PSI0Yml0IiwgY2hvaWNlcz1bIjRiaXQiLCAibm9uZSJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWZhbGxiYWNrIiwgZGVmYXVsdD0iUXdlbi9Rd2VuMi41LTNCLUluc3RydWN0IiwgaGVscD0idXNlZCB3aXRob3V0IHF1YW50aXphdGlvbiBpZiB0aGUgZmlyc3QgbW9kZWwgZmFpbHMgdG8gbG9hZCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2hfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbWF4X3dvcmRzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MjAwLCBoZWxwPSJlYWNoIHRleHQgaXMgY3V0IHRvIHRoaXMgbWFueSB3b3JkcyIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgaW1wb3J0IHRvcmNoCiAgICB3aXRoIG9wZW4oYXJncy5pdGVtcywgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBpdGVtcyA9IFtqc29uLmxvYWRzKGwpIGZvciBsIGluIGYgaWYgbC5zdHJpcCgpXQogICAgZG9uZSA9IHNldCgpCiAgICBpZiBvcy5wYXRoLmV4aXN0cyhhcmdzLm91dCk6CiAgICAgICAgd2l0aCBvcGVuKGFyZ3Mub3V0LCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBkb25lID0ge2pzb24ubG9hZHMobClbImlkIl0gZm9yIGwgaW4gZiBpZiBsLnN0cmlwKCl9CiAgICB0b2RvID0gW2l0IGZvciBpdCBpbiBpdGVtcyBpZiBpdFsiaWQiXSBub3QgaW4gZG9uZV0KICAgIHByaW50KGYie2xlbihpdGVtcyl9IGl0ZW1zLCB7bGVuKGRvbmUpfSBhbHJlYWR5IGp1ZGdlZCwge2xlbih0b2RvKX0gdG8gZ28iLCBmbHVzaD1UcnVlKQogICAgaWYgbm90IHRvZG86CiAgICAgICAgcmV0dXJuCgogICAgbW9kZWxfaWQsIHF1YW50ID0gYXJncy5tb2RlbCwgYXJncy5xdWFudGl6ZSA9PSAiNGJpdCIKICAgIHRyeToKICAgICAgICB0b2ssIG1vZGVsID0gbG9hZF9tb2RlbChtb2RlbF9pZCwgcXVhbnQsIHRvcmNoKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgIyBlLmcuIGJpdHNhbmRieXRlcyB1bmF2YWlsYWJsZQogICAgICAgIHByaW50KGYiY291bGQgbm90IGxvYWQge21vZGVsX2lkfSAoe3R5cGUoZSkuX19uYW1lX199OiB7ZX0pOyBmYWxsaW5nIGJhY2sgdG8ge2FyZ3MuZmFsbGJhY2t9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICBtb2RlbF9pZCwgcXVhbnQgPSBhcmdzLmZhbGxiYWNrLCBGYWxzZQogICAgICAgIHRvaywgbW9kZWwgPSBsb2FkX21vZGVsKG1vZGVsX2lkLCBxdWFudCwgdG9yY2gpCiAgICBtb2RlbF9sYWJlbCA9IG1vZGVsX2lkICsgKCIgKDQtYml0IE5GNCkiIGlmIHF1YW50IGVsc2UgIiAoZnAxNikiKQogICAgZGlnaXRzLCB5ZXNfaWRzLCBub19pZHMgPSBhbnN3ZXJfdG9rZW5faWRzKHRvaykKICAgIHByaW50KGYianVkZ2U6IHttb2RlbF9sYWJlbH0gfCBkaWdpdCB0b2tlbnMge2RpZ2l0c30gfCB5ZXMge3llc19pZHN9IHwgbm8ge25vX2lkc30iLCBmbHVzaD1UcnVlKQogICAgZGV2aWNlID0gbmV4dChtb2RlbC5wYXJhbWV0ZXJzKCkpLmRldmljZQogICAgZGVjb2RlciwgaGVhZCA9IGRlY29kZXJfYW5kX2hlYWQobW9kZWwpCgogICAgIyBsb25nZXN0IHByb21wdHMgZmlyc3QsIHNvIG1lbW9yeSBwcm9ibGVtcyBzaG93IHVwIGF0IG9uY2UKICAgIHRvZG8uc29ydChrZXk9bGFtYmRhIGl0OiAtbGVuKGJ1aWxkX3Byb21wdChpdCwgYXJncy5tYXhfd29yZHMpKSkKICAgIHQwID0gdGltZS50aW1lKCkKICAgIHdpdGggb3BlbihhcmdzLm91dCwgImEiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBvdXQsIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmb3IgYiBpbiByYW5nZSgwLCBsZW4odG9kbyksIGFyZ3MuYmF0Y2hfc2l6ZSk6CiAgICAgICAgICAgIGJhdGNoID0gdG9kb1tiOmIgKyBhcmdzLmJhdGNoX3NpemVdCiAgICAgICAgICAgIHRleHRzID0gW3Rvay5hcHBseV9jaGF0X3RlbXBsYXRlKFt7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBTWVNURU19LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IGJ1aWxkX3Byb21wdChpdCwgYXJncy5tYXhfd29yZHMpfV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSkgZm9yIGl0IGluIGJhdGNoXQogICAgICAgICAgICBlbmMgPSB0b2sodGV4dHMsIHJldHVybl90ZW5zb3JzPSJwdCIsIHBhZGRpbmc9VHJ1ZSkudG8oZGV2aWNlKQogICAgICAgICAgICBwcm9icyA9IHRvcmNoLnNvZnRtYXgobGFzdF9wb3NpdGlvbl9sb2dpdHMoZGVjb2RlciwgaGVhZCwgZW5jKSwgZGltPS0xKS5jcHUoKQogICAgICAgICAgICBmb3IgaXQsIHAgaW4gemlwKGJhdGNoLCBwcm9icyk6CiAgICAgICAgICAgICAgICByZWMgPSBkaWN0KGl0KQogICAgICAgICAgICAgICAgcmVjLnVwZGF0ZSh7Imp1ZGdlX21vZGVsIjogbW9kZWxfbGFiZWwsICJwcm9tcHRfdmVyc2lvbiI6IFBST01QVF9WRVJTSU9OLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgInRvcF90b2tlbiI6IHRvay5kZWNvZGUoW2ludChwLmFyZ21heCgpKV0pfSkKICAgICAgICAgICAgICAgIGlmIGl0WyJ0YXNrIl0gPT0gIm1lYW5pbmciOgogICAgICAgICAgICAgICAgICAgIHBkID0ge2Q6IGZsb2F0KHBbaV0pIGZvciBkLCBpIGluIGRpZ2l0cy5pdGVtcygpfQogICAgICAgICAgICAgICAgICAgIG1hc3MgPSBzdW0ocGQudmFsdWVzKCkpCiAgICAgICAgICAgICAgICAgICAgcmVjLnVwZGF0ZSh7InBfZGlnaXRzIjoge2Q6IHYgLyBtYXNzIGZvciBkLCB2IGluIHBkLml0ZW1zKCl9LCAiZm9ybWF0X21hc3MiOiBtYXNzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJleHBlY3RlZCI6IHN1bShpbnQoZCkgKiB2IGZvciBkLCB2IGluIHBkLml0ZW1zKCkpIC8gbWFzcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiYW5zd2VyIjogaW50KG1heChwZCwga2V5PXBkLmdldCkpfSkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgcHksIHBuID0gZmxvYXQocFt5ZXNfaWRzXS5zdW0oKSksIGZsb2F0KHBbbm9faWRzXS5zdW0oKSkKICAgICAgICAgICAgICAgICAgICByZWMudXBkYXRlKHsicF95ZXMiOiBweSAvIChweSArIHBuKSwgImZvcm1hdF9tYXNzIjogcHkgKyBwbiwgImFuc3dlciI6ICJ5ZXMiIGlmIHB5ID4gcG4gZWxzZSAibm8ifSkKICAgICAgICAgICAgICAgIGZvciBrIGluICgiYSIsICJiIiwgImluZm8iKToKICAgICAgICAgICAgICAgICAgICByZWMucG9wKGssIE5vbmUpICAjIGtlZXAgdGhlIHJlc3VsdHMgZmlsZSBmcmVlIG9mIHRoZSB0ZXh0cwogICAgICAgICAgICAgICAgb3V0LndyaXRlKGpzb24uZHVtcHMocmVjLCBlbnN1cmVfYXNjaWk9RmFsc2UpICsgIlxuIikKICAgICAgICAgICAgb3V0LmZsdXNoKCkKICAgICAgICAgICAgaWYgKGIgLy8gYXJncy5iYXRjaF9zaXplKSAlIDI1ID09IDA6CiAgICAgICAgICAgICAgICBwcmludChmIiAge2IgKyBsZW4oYmF0Y2gpfS97bGVuKHRvZG8pfSBqdWRnZWQgKHsodGltZS50aW1lKCkgLSB0MCkgLyA2MDouMWZ9IG1pbikiLCBmbHVzaD1UcnVlKQogICAgcHJpbnQoZiJmaW5pc2hlZCBpbiB7KHRpbWUudGltZSgpIC0gdDApIC8gNjA6LjFmfSBtaW4gd2l0aCB7bW9kZWxfbGFiZWx9IiwgZmx1c2g9VHJ1ZSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="
with open("judge_reviews/judge_items.jsonl", "wb") as f:
    f.write(gzip.decompress(base64.b64decode(ITEMS_B64)))
with open("llm_judge.py", "wb") as f:
    f.write(base64.b64decode(SRC_B64))
n = sum(1 for _ in open("judge_reviews/judge_items.jsonl", encoding="utf-8"))
print("items:", n)

In [ ]:
# Step 2 · Run the judge (resumable: rerunning this cell skips finished items)
import transformers
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "gpu": torch.cuda.get_device_name(0)}
try:
    import bitsandbytes
    env["bitsandbytes"] = bitsandbytes.__version__
except Exception as e:
    env["bitsandbytes"] = f"not available: {e}"
json.dump(env, open("judge_reviews/environment.json", "w"), indent=2)
print(env)
with open("judge_reviews/llm_judge.log", "a") as log:
    code = subprocess.run([sys.executable, "-u", "llm_judge.py", "--items", "judge_reviews/judge_items.jsonl",
                           "--out", "judge_reviews/judge_results.jsonl"], stdout=log, stderr=subprocess.STDOUT).returncode
print("exit code:", code)
print("".join(open("judge_reviews/llm_judge.log").readlines()[-15:]))
done = sum(1 for _ in open("judge_reviews/judge_results.jsonl", encoding="utf-8")) if os.path.exists("judge_reviews/judge_results.jsonl") else 0
print(f"judged {done} of {n} items")

In [ ]:
# Step 3 · Quick look, then download the outputs
import collections, statistics
res = [json.loads(l) for l in open("judge_reviews/judge_results.jsonl", encoding="utf-8")]
by = collections.defaultdict(list)
for r in res:
    by[(r["task"], r["method"])].append(r["expected"] if r["task"] == "meaning" else r["p_yes"])
for k, v in sorted(by.items()):
    print(k, len(v), round(statistics.mean(v), 3))
subprocess.run(["zip", "-qr", "reviews_judge_outputs.zip", "judge_reviews", "-x", "judge_reviews/judge_items.jsonl"], check=True)
from google.colab import files
files.download("reviews_judge_outputs.zip")
print("If the download doesn't start, open the Files pane (folder icon) and download reviews_judge_outputs.zip.")